# Generate an Antibody Binder with peleke-1

(Note: This notebook is designed for Google Colab. If you are running it locally, make sure to install the required dependencies and change the paths accordingly.)

In [ ]:
pip install torchao -U

## Download the model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, re

In [ ]:
model_name = 'silicobio/peleke-phi-4'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    trust_remote_code=True,
    ignore_mismatched_sizes=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)


## Create the antibody

peleke-1 model use annotated antigen sequences to produce targeted antibodies. In the string of amino acids in the antigen, we surround each epitope residue with square brackets (`[X]`). Then, we use a function to convert these to special `<epi>` and `</epi>` tokens.


In [ ]:
def format_prompt(antigen_sequence):
    epitope_seq = re.sub(r'\[([A-Z])\]', r'<epi>\1</epi>', antigen_sequence)
    formatted_str = f"Antigen: {epitope_seq}<|im_end|>\nAntibody:"
    return formatted_str

In [ ]:
## Example ## PD-1 antigen sequence with epitope residues marked
antigen_sequence = "NPPTFSPALLVVTEGDNATFTCSFS[S][F][V]L[N]WYRMQ[T][D][K]LAAF[P]E[D][R][S][Q][P][G]QDSRFRVTQLPNGRDFHMSVVRARRNDSGTYLCGA[I]S[L]AQIKESLRAELRV"

prompt = format_prompt(antigen_sequence)
inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.cuda() for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=False,
    )

In [ ]:
full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
antibody_sequence = full_text.split('<|im_end|>')[1].replace('Antibody: ', '')
print(f"Antigen: {antigen_sequence}\nAntibody: {antibody_sequence}\n")

## Evaluate the Antibody-Antigen Complex

Using your favorite folding tool, fold the antibody and antigen sequences together. Then, evaluate if the model produced an antibody that binds to the desired site.

In [ ]:
import py3Dmol

In [ ]:
cif_path = ''

with open(cif_path, "r") as f:
        cif_data = f.read()

view.addModel(cif_data, "cif", viewer=(r, c))

## Show binder in yellow
view.setStyle(
    {'chain':'A'},{'cartoon': {'color': 'yellow'}},
    viewer=(r, c)
)

## Show target in cyan
view.setStyle(
    {'chain':'B'},{'cartoon': {'color': 'cyan'}},
    viewer=(r, c)
)

view.zoomTo(viewer=(r, c))

## Try Again!

Now that you've seen how the *peleke-1* model works, find another target antigen from Protein Data Bank and try to design an antibody against it. You'll need to select the desired epitope residues and annotate them with `[X]`.